# Data Cleaning & Preprocessing 🧹

In [27]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [28]:
df = pd.read_csv("../data/raw/student_placement.csv")
df.shape

(100000, 18)

## Analysis: Raw Dataset Loading

### Observation:
- The raw dataset was successfully loaded into the cleaning notebook.
- The dataset contains **100,000 records and 18 columns**.
- The original raw dataset will be preserved, and cleaning operations will be performed on a separate working copy.

In [29]:
df.isnull().sum()

branch                           0
college_tier                     0
cgpa                             0
backlogs                         0
coding_skills                    0
dsa_score                        0
aptitude_score                   0
communication_skills             0
ml_knowledge                     0
system_design                    0
internships                      0
projects_count                   0
certifications                   0
hackathons                       0
open_source_contributions        0
extracurriculars                 0
placement_status                 0
salary_package_lpa           31525
dtype: int64

## Analysis: Missing Value Handling

### Observation:

- All predictor columns contain **0 missing values**.
- The `salary_package_lpa` column contains **31,525 missing values**.
- These missing values correspond to students whose `placement_status` is `0`.
- Since salary is only applicable after placement, these values are not treated as random missing data.

### Data Cleaning Decision:

- `salary_package_lpa` will be **excluded from the main placement prediction features**.
- The column will not be imputed because doing so would introduce artificial salary values.
- Excluding this column also prevents **data leakage**, since salary would not be known when predicting a student's placement probability.

In [30]:
df_clean = df.drop(columns=["salary_package_lpa"])
df_clean.shape

(100000, 17)

## Analysis: Removing Data Leakage Feature

### Observation:

- The original dataset contains 18 columns, including `salary_package_lpa`.
- `salary_package_lpa` was excluded from the model dataset because salary information is only available after placement.
- Including salary could introduce **data leakage** and lead to unrealistic model performance.
- The cleaned working dataset now contains **100,000 records and 17 columns**.
- The original 18-column raw dataset remains unchanged.

In [31]:
df_clean.dtypes

branch                           str
college_tier                     str
cgpa                         float64
backlogs                       int64
coding_skills                float64
dsa_score                    float64
aptitude_score               float64
communication_skills         float64
ml_knowledge                 float64
system_design                float64
internships                    int64
projects_count                 int64
certifications                 int64
hackathons                     int64
open_source_contributions      int64
extracurriculars               int64
placement_status               int64
dtype: object

## Analysis: Data Type Validation

### Observation:

- `branch` and `college_tier` are stored as string (`str`) values and are suitable for categorical encoding.
- The numerical features are stored as `int64` or `float64`.
- `placement_status` is stored as `int64` and contains the binary target values.
- No inappropriate or unexpected data types were identified.
- Therefore, no data-type conversion is required before proceeding with categorical encoding.

In [32]:
print("Branch categories:")
print(df_clean["branch"].unique())

print("\nCollege Tier categories:")
print(df_clean["college_tier"].unique())

Branch categories:
<ArrowStringArray>
['ECE', 'Chemical', 'EE', 'CE', 'CSE', 'IT', 'ME']
Length: 7, dtype: str

College Tier categories:
<ArrowStringArray>
['Tier-3', 'Tier-2', 'Tier-1']
Length: 3, dtype: str


## Analysis: Categorical Value Validation

### Observation:

- The `branch` column contains **7 valid categories**: ECE, Chemical, EE, CE, CSE, IT, and ME.
- The `college_tier` column contains **3 valid categories**: Tier-1, Tier-2, and Tier-3.
- No unexpected or invalid categorical values were identified.
- These categorical features are ready for encoding before model training.

In [33]:
df_clean["placement_status"].value_counts()

placement_status
1    68475
0    31525
Name: count, dtype: int64

## Analysis: Target Variable Validation

### Observation:

- The target variable `placement_status` contains two classes: `0` and `1`.
- There are **68,475 placed students (68.48%)** and **31,525 unplaced students (31.52%)**.
- Removing `salary_package_lpa` did not affect the target distribution.
- The target variable is suitable for **binary classification**.

In [34]:
X = df_clean.drop(columns=["placement_status"])
y = df_clean["placement_status"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (100000, 16)
Target shape: (100000,)


## Analysis: Feature and Target Separation

### Observation:

- The dataset contains **16 input features** used to predict placement.
- The target variable `placement_status` contains **100,000 observations**.
- `placement_status` is excluded from the input features to prevent the model from using the answer as an input.
- The feature matrix `X` has a shape of **(100000, 16)**, while the target vector `y` has a shape of **(100000,)**.

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (80000, 16)
X_test: (20000, 16)
y_train: (80000,)
y_test: (20000,)


## Analysis: Train-Test Split

### Observation:

- The dataset was divided into **80% training data and 20% testing data**.
- The training set contains **80,000 records**, while the testing set contains **20,000 records**.
- The target variable was stratified during splitting to maintain a similar placement distribution in both datasets.
- The training data will be used to train the machine learning models, while the testing data will be reserved for final model evaluation.

In [36]:
categorical_features = ["branch", "college_tier"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded X_train shape:", X_train_encoded.shape)
print("Encoded X_test shape:", X_test_encoded.shape)

Encoded X_train shape: (80000, 24)
Encoded X_test shape: (20000, 24)


## Analysis: Categorical Feature Encoding

### Observation:

- The categorical features `branch` and `college_tier` were encoded using **One-Hot Encoding**.
- `branch` was converted into **7 numerical features**.
- `college_tier` was converted into **3 numerical features**.
- The 14 numerical features were retained without modification.
- The feature count increased from **16 to 24** after encoding.
- The encoded training dataset contains **80,000 records and 24 features**, while the encoded testing dataset contains **20,000 records and 24 features**.
- The encoder was fitted only on the training data and then applied to the testing data to prevent data leakage.

In [37]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))
print("\nEncoded feature names:")
print(feature_names)

Number of features: 24

Encoded feature names:
['categorical__branch_CE' 'categorical__branch_CSE'
 'categorical__branch_Chemical' 'categorical__branch_ECE'
 'categorical__branch_EE' 'categorical__branch_IT'
 'categorical__branch_ME' 'categorical__college_tier_Tier-1'
 'categorical__college_tier_Tier-2' 'categorical__college_tier_Tier-3'
 'remainder__cgpa' 'remainder__backlogs' 'remainder__coding_skills'
 'remainder__dsa_score' 'remainder__aptitude_score'
 'remainder__communication_skills' 'remainder__ml_knowledge'
 'remainder__system_design' 'remainder__internships'
 'remainder__projects_count' 'remainder__certifications'
 'remainder__hackathons' 'remainder__open_source_contributions'
 'remainder__extracurriculars']


## Analysis: Encoded Feature Verification

### Observation:

- The preprocessing pipeline generated **24 features** from the original 16 input features.
- The `branch` feature was converted into **7 one-hot encoded features**.
- The `college_tier` feature was converted into **3 one-hot encoded features**.
- The remaining **14 numerical features** were retained.
- The generated feature names confirm that the categorical variables were encoded correctly.
- No target variable or salary information is present among the encoded features.

In [38]:
print("Training data:")
print("Shape:", X_train_encoded.shape)
print("Contains NaN:", np.isnan(X_train_encoded).any())

print("\nTesting data:")
print("Shape:", X_test_encoded.shape)
print("Contains NaN:", np.isnan(X_test_encoded).any())

print("\nTarget distribution:")
print(y_train.value_counts())

Training data:
Shape: (80000, 24)
Contains NaN: False

Testing data:
Shape: (20000, 24)
Contains NaN: False

Target distribution:
placement_status
1    54780
0    25220
Name: count, dtype: int64


## Analysis: Final Preprocessing Validation

### Observation:

- The training dataset contains **80,000 records and 24 features**.
- The testing dataset contains **20,000 records and 24 features**.
- No missing (`NaN`) values are present in either the training or testing features.
- The target variable contains both placement classes after the train-test split.
- The preprocessing pipeline has successfully prepared the data for machine learning model training.
- The training and testing datasets have the same feature structure, ensuring compatibility during model evaluation.

In [39]:
df_clean.to_csv(
    "../data/processed/student_placement_clean.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


## Final Step: Save Cleaned Dataset

### Observation:

- The cleaned dataset was successfully saved in the `data/processed` directory.
- The processed dataset contains **100,000 records and 17 columns**.
- The original raw dataset remains unchanged.
- The processed dataset will be used as the input for **Exploratory Data Analysis and subsequent project phases**.